In [0]:
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

import os
# env = os.environ["S3PREFIX"] 
# Import the module
import sys
sys.path.append("../aws_utils")
 
import aws_utils
# Get Environment Prefix
 
env = aws_utils.get_env_prefix()
import delta_table_creator, transform

In [0]:
schema = (
    "DEntityRelationshipId     string,"
    "SourceSystem              string,"
    "PortfolioKey              string,"
    "PortfolioName             string,"
    "IngestionTimestamp        timestamp,"
    "BatchId                   string,"
    "DEntityRelationshipKey    string,"
    "EntityKey                 string,"
    "RelatedEntityKey          string,"
    "EntityRelationshipType    string,"
    "EntityRelationshipSubType string,"
    "EntityType                string,"
    "RelatedEntityType         string,"
    "BusinessDate              date"
)

partition_columns = ["BusinessDate"]
run_date = dbutils.widgets.get("run_date")
unity = dbutils.widgets.get("unity")
table_name = "DEntityRelationships"
derivative_table_path = ""

# ================== Create Delta Table ==================
if unity == "True":
    derivative_table_path = f"`{env}-cdh`.derivativeactive.{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_unity_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )
else:
    derivative_table_path = f"s3://{env}-ec1-cepal-derivative-zone/{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )


# ================== Delete From Delta Table ==================

transform.delete_from_unity_table(
    spark,
    derivative_table_path,
    run_date,  # type: ignore
)

## **DEntityRelationships_profits**

In [0]:
# Create temp view for each source table, based on the run date

EntityRelationships_profits = spark.sql(f""" select 
    EntityRelationshipId,
    SourceSystem,
    PortfolioKey,
    PortfolioName,
    IngestionTimestamp,
    BatchId,
    EntityRelationshipKey,
    EntityKey,
    RelatedEntityKey,
    EntityRelationshipType,
    EntityRelationshipSubType,
    EntityType,
    RelatedEntityType,
    BusinessDate
FROM  `{env}-cdh`.primary.EntityRelationships
WHERE BusinessDate = {run_date!r} AND SourceSystem="profits"
"""
) 

EntityRelationships_profits.createOrReplaceTempView("EntityRelationships_profits")

In [0]:
target_table_dentityrelationshipsId_primary_key_fields = [
    EntityRelationships_profits.EntityRelationshipId,
]

target_table_dentityrelationshipsKey_primary_key_fields = [
    EntityRelationships_profits.EntityRelationshipKey,
]

target_table_dentityrelationships_profits = (
    EntityRelationships_profits.withColumn(
        "DEntityRelationshipId",
        sha2(
            concat_ws(";", *target_table_dentityrelationshipsId_primary_key_fields),
            256,
        ),
    ).withColumn(
        "DEntityRelationshipKey",
        concat_ws(";", *target_table_dentityrelationshipsKey_primary_key_fields)
    )
)

dentityrelationships_profits = target_table_dentityrelationships_profits.select(
    target_table_dentityrelationships_profits.DEntityRelationshipId.cast("String"),
    target_table_dentityrelationships_profits.SourceSystem.cast("String"),
    target_table_dentityrelationships_profits.PortfolioKey.cast("String"),
    target_table_dentityrelationships_profits.PortfolioName.cast("String"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    target_table_dentityrelationships_profits.DEntityRelationshipKey.cast("String"),
    target_table_dentityrelationships_profits.EntityKey.cast("String"),
    target_table_dentityrelationships_profits.RelatedEntityKey.cast("String"),
    target_table_dentityrelationships_profits.EntityRelationshipType.cast("String"),
    target_table_dentityrelationships_profits.EntityRelationshipSubType.cast("String"),
    target_table_dentityrelationships_profits.EntityType.cast("String"),
    target_table_dentityrelationships_profits.RelatedEntityType.cast("String"),
    target_table_dentityrelationships_profits.BusinessDate,
)

## **DEntityRelationships_qca**

In [0]:
# Create temp view for each source table, based on the run date

EntityRelationships_qca = spark.sql(f""" select 
    EntityRelationshipId,
    SourceSystem,
    PortfolioKey,
    PortfolioName,
    IngestionTimestamp,
    BatchId,
    EntityRelationshipKey,
    EntityKey,
    RelatedEntityKey,
    EntityRelationshipType,
    EntityRelationshipSubType,
    EntityType,
    RelatedEntityType,
    BusinessDate
FROM  `{env}-cdh`.primary.EntityRelationships
WHERE BusinessDate = {run_date!r} AND SourceSystem="qca"
"""
) 

EntityRelationships_qca.createOrReplaceTempView("EntityRelationships_qca")

In [0]:
target_table_DEntityRelationships_qca = spark.sql(f""" SELECT
    ent_qca.EntityRelationshipId,
    ent_qca.SourceSystem,
    ent_qca.PortfolioKey,
    ent_qca.PortfolioName,
    ent_qca.IngestionTimestamp,
    ent_qca.BatchId,
    ent_qca.EntityRelationshipKey,
    ent_qca.EntityKey,
    ent_qca.RelatedEntityKey,
    ent_qca.EntityRelationshipType,
    ent_qca.EntityRelationshipSubType,
    ent_qca.EntityType,
    ent_qca.RelatedEntityType,
    ent_qca.BusinessDate
FROM EntityRelationships_qca ent_qca
left join EntityRelationships_profits ent_profits
on substring(ent_profits.EntityKey, 4, 50) = ent_qca.EntityKey
and substring(ent_profits.RelatedEntityKey, 4, 50) = ent_qca.RelatedEntityKey
where ent_profits.EntityKey is null
""")

In [0]:
target_table_dentityrelationshipsid_primary_key_fields = [
    target_table_DEntityRelationships_qca.EntityRelationshipId
]

target_table_dentityrelationshipsKey_primary_key_fields = [
    target_table_DEntityRelationships_qca.EntityRelationshipKey
]

target_table_dentityrelationships_qca = (
    target_table_DEntityRelationships_qca.withColumn(
        "DEntityRelationshipId",
        sha2(
            concat_ws(";", *target_table_dentityrelationshipsid_primary_key_fields),
            256,
        ),
    ).withColumn(
        "DEntityRelationshipKey",
        concat_ws(";", *target_table_dentityrelationshipsKey_primary_key_fields)
    )
)

dentityrelationships_qca = target_table_dentityrelationships_qca.select(
    target_table_dentityrelationships_qca.DEntityRelationshipId.cast("String"),
    target_table_dentityrelationships_qca.SourceSystem.cast("String"),
    target_table_dentityrelationships_qca.PortfolioKey.cast("String"),
    target_table_dentityrelationships_qca.PortfolioName.cast("String"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    target_table_dentityrelationships_qca.DEntityRelationshipKey.cast("String"),
    target_table_dentityrelationships_qca.EntityKey.cast("String"),
    target_table_dentityrelationships_qca.RelatedEntityKey.cast("String"),
    target_table_dentityrelationships_qca.EntityRelationshipType.cast("String"),
    target_table_dentityrelationships_qca.EntityRelationshipSubType.cast("String"),
    target_table_dentityrelationships_qca.EntityType.cast("String"),
    target_table_dentityrelationships_qca.RelatedEntityType.cast("String"),
    target_table_dentityrelationships_qca.BusinessDate,
)

## **DEntityRelationships_arotron**

In [0]:
# Create temp view for each source table, based on the run date

EntityRelationships_arotron = spark.sql(f""" select 
    EntityRelationshipId,
    SourceSystem,
    PortfolioKey,
    PortfolioName,
    IngestionTimestamp,
    BatchId,
    EntityRelationshipKey,
    EntityKey,
    RelatedEntityKey,
    EntityRelationshipType,
    EntityRelationshipSubType,
    EntityType,
    RelatedEntityType,
    BusinessDate
FROM  `{env}-cdh`.primary.EntityRelationships
WHERE BusinessDate = {run_date!r} AND SourceSystem="arotron"
"""
) 

EntityRelationships_arotron.createOrReplaceTempView("EntityRelationships_arotron")

In [0]:
target_table_DEntityRelationships_arotron = spark.sql(f""" SELECT
    ent_arotron.EntityRelationshipId,
    ent_arotron.SourceSystem,
    ent_arotron.PortfolioKey,
    ent_arotron.PortfolioName,
    ent_arotron.IngestionTimestamp,
    ent_arotron.BatchId,
    ent_arotron.EntityRelationshipKey,
    ent_arotron.EntityKey,
    ent_arotron.RelatedEntityKey,
    ent_arotron.EntityRelationshipType,
    ent_arotron.EntityRelationshipSubType,
    ent_arotron.EntityType,
    ent_arotron.RelatedEntityType,
    ent_arotron.BusinessDate
FROM EntityRelationships_arotron ent_arotron
left join EntityRelationships_profits ent_profits
on substring(ent_profits.EntityKey, 4, 50) = ent_arotron.EntityKey
and substring(ent_profits.RelatedEntityKey, 4, 50) = ent_arotron.RelatedEntityKey
where ent_profits.EntityKey is null
""")

In [0]:
target_table_dentityrelationshipsid_primary_key_fields = [
    target_table_DEntityRelationships_arotron.EntityRelationshipId
]

target_table_dentityrelationshipsKey_primary_key_fields = [
    target_table_DEntityRelationships_arotron.EntityRelationshipKey
]

target_table_dentityrelationships_arotron = (
    target_table_DEntityRelationships_arotron.withColumn(
        "DEntityRelationshipId",
        sha2(
            concat_ws(";", *target_table_dentityrelationshipsid_primary_key_fields),
            256,
        ),
    ).withColumn(
        "DEntityRelationshipKey",
        concat_ws(";", *target_table_dentityrelationshipsKey_primary_key_fields)
    )
)

dentityrelationships_arotron = target_table_dentityrelationships_arotron.select(
    target_table_dentityrelationships_arotron.DEntityRelationshipId.cast("String"),
    target_table_dentityrelationships_arotron.SourceSystem.cast("String"),
    target_table_dentityrelationships_arotron.PortfolioKey.cast("String"),
    target_table_dentityrelationships_arotron.PortfolioName.cast("String"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    target_table_dentityrelationships_arotron.DEntityRelationshipKey.cast("String"),
    target_table_dentityrelationships_arotron.EntityKey.cast("String"),
    target_table_dentityrelationships_arotron.RelatedEntityKey.cast("String"),
    target_table_dentityrelationships_arotron.EntityRelationshipType.cast("String"),
    target_table_dentityrelationships_arotron.EntityRelationshipSubType.cast("String"),
    target_table_dentityrelationships_arotron.EntityType.cast("String"),
    target_table_dentityrelationships_arotron.RelatedEntityType.cast("String"),
    target_table_dentityrelationships_arotron.BusinessDate,
)

## Final DataFrame DEntityRelationships

In [0]:
dentityrelationships=dentityrelationships_profits.union(dentityrelationships_qca).union(dentityrelationships_arotron)

In [0]:
if unity == "True":
    dentityrelationships.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(derivative_table_path) # in order to write in delta_lake
else: 
    dentityrelationships.write \
        .format("delta") \
        .mode("append") \
        .save(derivative_table_path) # in order to write in delta_lake 